In [0]:
-- ==============================================================================
-- Gold Layer: Dimensional Modeling (Star Schema)
-- Source: ecommerce_silver.cleansed_sales
-- Target Schema: ecommerce_gold
-- ==============================================================================

-- 1. Create Gold Schema
CREATE SCHEMA IF NOT EXISTS ecommerce_gold;

In [0]:
-- ------------------------------------------------------------------------------
-- 2. DIMENSION: dim_customers
-- ------------------------------------------------------------------------------
CREATE OR REPLACE TABLE ecommerce_gold.dim_customers
USING DELTA
AS
SELECT DISTINCT
    customer_key,
    customer_id,
    CURRENT_TIMESTAMP() AS created_at
FROM ecommerce_silver.cleansed_sales;


In [0]:
-- ------------------------------------------------------------------------------
-- 3. DIMENSION: dim_products
-- ------------------------------------------------------------------------------
CREATE OR REPLACE TABLE ecommerce_gold.dim_products
USING DELTA
AS
SELECT DISTINCT
    MD5(product_category) AS product_key,
    product_category,
    CURRENT_TIMESTAMP() AS created_at
FROM ecommerce_silver.cleansed_sales;


In [0]:
-- ------------------------------------------------------------------------------
-- 4. DIMENSION: dim_date
-- Dynamic Date Table generated from Silver order_date range
-- ------------------------------------------------------------------------------
CREATE OR REPLACE TABLE ecommerce_gold.dim_date
USING DELTA
AS
WITH date_range AS (
    SELECT 
        MIN(order_date) AS min_date,
        MAX(order_date) AS max_date
    FROM ecommerce_silver.cleansed_sales
),
dates AS (
    SELECT 
        EXPLODE(SEQUENCE(min_date, max_date, INTERVAL 1 DAY)) AS full_date
    FROM date_range
)
SELECT 
    CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT) AS date_key,
    full_date                                       AS order_date,
    YEAR(full_date)                                 AS year,
    QUARTER(full_date)                              AS quarter,
    MONTH(full_date)                                AS month,
    DATE_FORMAT(full_date, 'MMMM')                  AS month_name,
    DAY(full_date)                                  AS day,
    DAYOFWEEK(full_date)                            AS day_of_week,
    DATE_FORMAT(full_date, 'EEEE')                  AS day_name
FROM dates;

In [0]:
-- ------------------------------------------------------------------------------
-- 5. FACT TABLE: fact_sales
-- ------------------------------------------------------------------------------
CREATE OR REPLACE TABLE ecommerce_gold.fact_sales
USING DELTA
AS
SELECT 
    s.order_id,
    
    -- Dimensional Foreign Keys
    s.customer_key,
    MD5(s.product_category)                              AS product_key,
    CAST(DATE_FORMAT(s.order_date, 'yyyyMMdd') AS INT)   AS date_key,
    
    -- Degenerate Dimensions / Categoricals
    s.region,
    s.payment_method,
    
    -- Measures / Metrics
    s.quantity,
    s.unit_price,
    s.discount,
    s.revenue,
    s.delivery_days,
    s.customer_rating,
    
    -- Metadata
    CURRENT_TIMESTAMP() AS created_at

FROM ecommerce_silver.cleansed_sales s;

In [0]:
-- ==============================================================================
-- GOLD LAYER DATA QUALITY CHECKS
-- ==============================================================================

-- ------------------------------------------------------------------------------
-- CHECK 1: Volumetric & Revenue Reconciliation (Silver vs. Gold)
-- Objective: Ensure no records or revenue were lost/duplicated during ETL
-- ------------------------------------------------------------------------------

SELECT 
    'Silver (cleansed_sales)' AS layer, 
    COUNT(*) AS total_records, 
    SUM(revenue) AS total_revenue
FROM ecommerce_silver.cleansed_sales
UNION ALL
SELECT 
    'Gold (fact_sales)' AS layer, 
    COUNT(*) AS total_records, 
    SUM(revenue) AS total_revenue
FROM ecommerce_gold.fact_sales;

-- Expected Outcome: Both rows must show the exact same count and sum.

In [0]:
-- ------------------------------------------------------------------------------
-- CHECK 2: Primary Key Uniqueness in Dimensions
-- Objective: Confirm dimensions have 100% unique primary keys (no duplicates)
-- ------------------------------------------------------------------------------

SELECT 'dim_customers' AS dimension, COUNT(*) - COUNT(DISTINCT customer_key) AS duplicate_keys FROM ecommerce_gold.dim_customers
UNION ALL
SELECT 'dim_products' AS dimension, COUNT(*) - COUNT(DISTINCT product_key) AS duplicate_keys FROM ecommerce_gold.dim_products
UNION ALL
SELECT 'dim_date' AS dimension, COUNT(*) - COUNT(DISTINCT date_key) AS duplicate_keys FROM ecommerce_gold.dim_date;

-- Expected Outcome: duplicate_keys must be 0 for all rows.

In [0]:
-- ------------------------------------------------------------------------------
-- CHECK 3: Referential Integrity (Orphan Foreign Keys in Fact Table)
-- Objective: Detect any fact row referencing non-existent dimension keys
-- ------------------------------------------------------------------------------

SELECT 
    COUNT(CASE WHEN c.customer_key IS NULL THEN 1 END) AS missing_customers,
    COUNT(CASE WHEN p.product_key IS NULL THEN 1 END)  AS missing_products,
    COUNT(CASE WHEN d.date_key IS NULL THEN 1 END)     AS missing_dates
FROM ecommerce_gold.fact_sales f
LEFT JOIN ecommerce_gold.dim_customers c ON f.customer_key = c.customer_key
LEFT JOIN ecommerce_gold.dim_products p  ON f.product_key = p.product_key
LEFT JOIN ecommerce_gold.dim_date d      ON f.date_key = d.date_key;

-- Expected Outcome: All missing_* counts must be 0.

In [0]:
-- ------------------------------------------------------------------------------
-- CHECK 4: Null Key Check in Fact Table
-- Objective: Confirm no Foreign Keys in fact_sales are NULL
-- ------------------------------------------------------------------------------

SELECT 
    COUNT(*) AS total_null_keys
FROM ecommerce_gold.fact_sales
WHERE customer_key IS NULL 
   OR product_key IS NULL 
   OR date_key IS NULL;
   
-- Expected Outcome: total_null_keys must be 0.

In [0]:
SELECT * 
FROM ecommerce_gold.fact_sales 
LIMIT 10;